<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/Prompt_Induced_Bias_in_Code_Switching_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# --- 1. MOCK LLM DATA GENERATION FUNCTION ---
# This function simulates the LLM output based on the prompt's instructions.
def mock_llm_generator(prompt_template, N=500):
    """Simulates generating code-switched data with potential bias."""
    data = []
    np.random.seed(42)

    # Define the attributes that the prompt forces the LLM to use
    genders = ['male', 'female']
    professions = ['software engineer', 'nurse']

    # Simple probability map based on the *prompt's intent*
    # 0: Low-paying/Stereotypical (Nurse, Service), 1: High-paying/Technical (Engineer)
    profession_map = {'software engineer': 1, 'nurse': 0}

    for i in range(N):
        # We assume the LLM cycles through or randomly selects a prompt
        gender = genders[i % len(genders)]

        # Determine the profession based on the prompt template used
        if "STEREOTYPED" in prompt_template:
            # Biased Logic: Male -> Engineer (High-status), Female -> Nurse (Low-status)
            profession = 'software engineer' if gender == 'male' else 'nurse'
        else:
            # Neutral/Debiased Logic: Randomly assign profession regardless of gender
            profession = np.random.choice(professions)

        # Simulate a code-switched sentence feature (e.g., length, complexity)
        # Assume technical professions (1) tend to generate slightly longer sentences (just a mock feature)
        sentence_feature = np.random.normal(50 + 10 * profession_map[profession], 10)

        data.append({
            'Gender': gender,
            'Profession': profession,
            'Sentence_Feature': sentence_feature,
            'Generated_Prompt_Type': prompt_template
        })
    return pd.DataFrame(data)

In [ ]:
# --- 2. BIAS INTRODUCED VIA THE PROMPT ---

# Initial, Biased Prompt Template (Implicitly directs the LLM to reinforce stereotypes)
BIASED_PROMPT = "STEREOTYPED: Generate a code-switched Spanish/English sentence about a {gender} working as a {profession}. Ensure the sentence is complex."

# Generate Biased Data (500 samples)
df_biased = mock_llm_generator(BIASED_PROMPT, N=500)

print("##  BIAS ANALYSIS: Resulting Data from Biased Prompt ##")
print("> The prompt forces a correlation between gender and profession.")
# Show the correlation (Profession distribution by Gender)
print(pd.crosstab(df_biased['Gender'], df_biased['Profession'], normalize='index'))
print("-" * 60)

##  BIAS ANALYSIS: Resulting Data from Biased Prompt ##
> The prompt forces a correlation between gender and profession.
Profession  nurse  software engineer
Gender                              
female        1.0                0.0
male          0.0                1.0
------------------------------------------------------------


In [ ]:
# --- 3. MITIGATION VIA PROMPT CHANGE ---

# Debiased Prompt Template (Explicitly requests balanced assignments)
DEBIASED_PROMPT = "BALANCED: Generate a code-switched Spanish/English sentence about a {gender} working in a *randomly assigned* profession. Ensure the dataset covers equal examples of 'software engineer' and 'nurse' for all genders."

# Generate Debiased Data (500 samples)
df_debiased = mock_llm_generator(DEBIASED_PROMPT, N=500)

print("##  MITIGATION ANALYSIS: Resulting Data from Debiased Prompt ##")
print("> The revised prompt explicitly enforces diversity, breaking the correlation.")
# Show the new, mitigated correlation
print(pd.crosstab(df_debiased['Gender'], df_debiased['Profession'], normalize='index'))
print("-" * 60)

##  MITIGATION ANALYSIS: Resulting Data from Debiased Prompt ##
> The revised prompt explicitly enforces diversity, breaking the correlation.
Profession  nurse  software engineer
Gender                              
female      0.468              0.532
male        0.448              0.552
------------------------------------------------------------
